In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.utils.class_weight import compute_class_weight

In [2]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)
 
DATA_DIR = "../data/"
MAX_WORDS = 20000      
MAX_LEN = 50           
EMBED_DIM = 128
USE_CLASS_WEIGHT = True

In [3]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
val = pd.read_csv(f"{DATA_DIR}/val.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
 
print("Shapes:", train.shape, val.shape, test.shape)
print("\nTrain label dagilimi:\n", train["label"].value_counts().sort_index())

Shapes: (16000, 2) (2000, 2) (2000, 2)

Train label dagilimi:
 label
0    4666
1    5362
2    1304
3    2159
4    1937
5     572
Name: count, dtype: int64


In [4]:
conflicts = train.groupby("text")["label"].nunique()
print("\nAyni metin farkli etiketle geciyor mu?:", (conflicts > 1).sum())
train = train.drop_duplicates().reset_index(drop=True)
print("Duplicate temizligi sonrasi train:", train.shape)


Ayni metin farkli etiketle geciyor mu?: 30
Duplicate temizligi sonrasi train: (15999, 2)


In [5]:
overlap = set(test["text"]) & set(train["text"])
print(f"Test'te train ile ayni metin sayisi: {len(overlap)}")
 
X_train_text, y_train = train["text"].values, train["label"].values
X_val_text, y_val = val["text"].values, val["label"].values
X_test_text, y_test = test["text"].values, test["label"].values

Test'te train ile ayni metin sayisi: 11


In [6]:
tokenizer = keras.preprocessing.text.Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train_text)

In [7]:
def encode(texts):
    seqs = tokenizer.texts_to_sequences(texts)
    return keras.preprocessing.sequence.pad_sequences(
        seqs, maxlen=MAX_LEN, padding="post", truncating="post"
    )

In [8]:
X_train = encode(X_train_text)
X_val = encode(X_val_text)
X_test = encode(X_test_text)
 
print("\nX_train shape:", X_train.shape)
print("Ornek token dizisi:", X_train[0][:12])


X_train shape: (15999, 50)
Ornek token dizisi: [  2 139   3 679   0   0   0   0   0   0   0   0]


In [9]:
NUM_CLASSES = len(np.unique(y_train))

In [12]:
RNN_UNITS=64
model = keras.Sequential([
    layers.Input(shape=(MAX_LEN,)),
    layers.Embedding(input_dim=MAX_WORDS, output_dim=EMBED_DIM, mask_zero=True),
    layers.Bidirectional(layers.SimpleRNN(RNN_UNITS)),   # LSTM yerine SimpleRNN
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax"),
])

In [13]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",   # label tam sayi (0-5) oldugu icin
    metrics=["accuracy"],
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)              │ (None, 50, 128)             │       2,560,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ bidirectional (Bidirectional)        │ (None, 128)                 │          24,704 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 6)                   │             390 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,593,350 (9.89 MB)

 Trainable params: 2,593,350 (9.89 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
class_weight = None
if USE_CLASS_WEIGHT:
    weights = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weight = dict(enumerate(weights))
    print("\nClass weights:", {k: round(v, 2) for k, v in class_weight.items()})


Class weights: {0: np.float64(0.57), 1: np.float64(0.5), 2: np.float64(2.04), 3: np.float64(1.24), 4: np.float64(1.38), 5: np.float64(4.66)}


In [15]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=3, restore_best_weights=True
)

In [16]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=64,
    class_weight=class_weight,
    callbacks=[early_stop],
    verbose=2,
)
 

Epoch 1/30
250/250 - 33s - 133ms/step - accuracy: 0.2191 - loss: 1.7222 - val_accuracy: 0.4170 - val_loss: 1.4074
Epoch 2/30
250/250 - 39s - 158ms/step - accuracy: 0.6969 - loss: 0.7790 - val_accuracy: 0.7995 - val_loss: 0.6062
Epoch 3/30
250/250 - 21s - 83ms/step - accuracy: 0.9056 - loss: 0.2544 - val_accuracy: 0.8415 - val_loss: 0.5080
Epoch 4/30
250/250 - 21s - 83ms/step - accuracy: 0.9537 - loss: 0.1173 - val_accuracy: 0.8055 - val_loss: 0.6389
Epoch 5/30
250/250 - 20s - 79ms/step - accuracy: 0.9711 - loss: 0.0756 - val_accuracy: 0.8425 - val_loss: 0.5587
Epoch 6/30
250/250 - 19s - 75ms/step - accuracy: 0.9821 - loss: 0.0507 - val_accuracy: 0.8290 - val_loss: 0.6216


In [17]:
y_prob = model.predict(X_test, verbose=0)
y_pred = y_prob.argmax(axis=1)
 
print("\n=== RNN Test Sonuclari ===")
print(f"Accuracy : {(y_pred == y_test).mean():.4f}")
print(f"Macro F1 : {f1_score(y_test, y_pred, average='macro'):.4f}")
print(classification_report(y_test, y_pred, digits=3))
print("Confusion matrix (satir=gercek, sutun=tahmin):")
print(confusion_matrix(y_test, y_pred))


=== RNN Test Sonuclari ===
Accuracy : 0.8345
Macro F1 : 0.7828
              precision    recall  f1-score   support

           0      0.866     0.893     0.880       581
           1      0.913     0.829     0.869       695
           2      0.652     0.862     0.743       159
           3      0.804     0.789     0.796       275
           4      0.814     0.799     0.806       224
           5      0.586     0.621     0.603        66

    accuracy                          0.835      2000
   macro avg      0.772     0.799     0.783      2000
weighted avg      0.842     0.835     0.836      2000

Confusion matrix (satir=gercek, sutun=tahmin):
[[519  18   5  31   8   0]
 [ 34 576  61   8   6  10]
 [ 14   5 137   1   0   2]
 [ 17  17   2 217  19   3]
 [ 12   4   4  11 179  14]
 [  3  11   1   2   8  41]]


In [18]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_tr_tfidf = tfidf.fit_transform(X_train_text)
X_te_tfidf = tfidf.transform(X_test_text)
 
lr = LogisticRegression(max_iter=2000, class_weight="balanced")
lr.fit(X_tr_tfidf, y_train)
lr_pred = lr.predict(X_te_tfidf)
 
print("\n=== Baseline (TF-IDF + LogReg) ===")
print(f"Accuracy : {(lr_pred == y_test).mean():.4f}")
print(f"Macro F1 : {f1_score(y_test, lr_pred, average='macro'):.4f}")


=== Baseline (TF-IDF + LogReg) ===
Accuracy : 0.8690
Macro F1 : 0.8345


In [19]:
model.save("../model/tweet_emotions.keras")